In [1]:
import re
import json
import os
import yfinance as yf
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import CodeInterpreterTool, FileReadTool


In [4]:

from dotenv import load_dotenv

load_dotenv(override=True)


True

In [5]:

class QueryAnalysisOutput(BaseModel):
    """Structured output for the query analysis task."""
    symbols: list[str] = Field(..., description="List of stock ticker symbols (e.g., ['TSLA', 'AAPL']).")
    timeframe: str = Field(..., description="Time period (e.g., '1d', '1mo', '1y').")
    action: str = Field(..., description="Action to be performed (e.g., 'fetch', 'plot').")


In [6]:
os.getenv("LLM_MODEL")

'openai/openai-main/gpt-4o-mini'

In [7]:
os.getenv("LLM_GATEWAY_URL")

'https://internal.devtest.truefoundry.tech/api/llm'

In [8]:
os.getenv("TFY_API_KEY")

'eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6ImxzV0lDNWtkU1V1bXg1ckg5NkR6bFdYUGxJTSJ9.eyJhdWQiOiI4OTUyNTNhZi1lYzlkLTRiZTYtODNkMS02ZjI0OGU2NDRlNzkiLCJleHAiOjM3MTQxMDMyOTMsImlhdCI6MTc1NDU1MTI5MywiaXNzIjoidHJ1ZWZvdW5kcnkuY29tIiwic3ViIjoiY21lMTJqZThsMGdrZDAxc2VjazZ1ZTNqYiIsImp0aSI6IjdiZjZiMmEwLWMwMTQtNGQ0Yy1hNmU4LTU2Y2RkMjJhZDJlOCIsInN1YmplY3RTbHVnIjoiZGVmYXVsdC1jbWR4MWdteWMwMGdtMDFuODJxeno4Z2sxIiwidXNlcm5hbWUiOiJkZWZhdWx0LWNtZHgxZ215YzAwZ20wMW44MnF6ejhnazEiLCJ1c2VyVHlwZSI6InNlcnZpY2VhY2NvdW50Iiwic3ViamVjdFR5cGUiOiJzZXJ2aWNlYWNjb3VudCIsInRlbmFudE5hbWUiOiJ0cnVlZm91bmRyeSIsInJvbGVzIjpbXSwiYXBwbGljYXRpb25JZCI6Ijg5NTI1M2FmLWVjOWQtNGJlNi04M2QxLTZmMjQ4ZTY0NGU3OSJ9.hx6yn1faXSkh7Wbe0eyKNka_rAdQ746MQHerhst2Npd7osmCpwnKAAVc3Mrv-CIHtTA25oYRKRGXhIDsc8JqoanRtgW5Tp2JgjrKCjRntRqyoF2ohB9x2SmP4G5PfJJ0fs3ZfwLKbz2BKgFGI-_wltPba1LDiDaYwodmI7zUfsoyJNNBt7ipy_b5cQpb3dWJkBqMQXTvHLkHk9-PN5fjD6Pmd6XxFCMVSlZyaD3AkL9NbN3DCJJBYDc7x7UD9frUAKhp5ehggtekJMsqa55ELR55sVggqTmltDLvlu9APCD6F0LFnw9woaWAbmagaKJfFGQccTPsh1VvXdEiX

In [9]:

llm = LLM(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_GATEWAY_URL"),
    api_key=os.getenv("TFY_API_KEY")
    # temperature=0.7
)


In [10]:

# 1) Query parser agent
query_parser_agent = Agent(
    role="Stock Data Analyst",
    goal="Extract stock details and fetch required data from this user query: {query}.",
    backstory="You are a financial analyst specializing in stock market data retrieval.",
    llm=llm,
    verbose=True,
    memory=True,
)


In [11]:

query_parsing_task = Task(
    description="Analyze the user query and extract stock details.",
    expected_output="A dictionary with keys: 'symbol', 'timeframe', 'action'.",
    output_pydantic=QueryAnalysisOutput,
    agent=query_parser_agent,
)


In [12]:


# 2) Code writer agent
code_writer_agent = Agent(
    role="Senior Python Developer",
    goal="Write Python code to visualize stock data.",
    backstory="""You are a Senior Python developer specializing in stock market data visualization. 
                 You are also a Pandas, Matplotlib and yfinance library expert.
                 You are skilled at writing production-ready Python code""",
    llm=llm,
    verbose=True,
)


In [13]:

code_writer_task = Task(
    description="""Write Python code to visualize stock data based on the inputs from the stock analyst
                   where you would find stock symbol, timeframe and action.""",
    expected_output="A clean and executable Python script file (.py) for stock visualization.",
    agent=code_writer_agent,
)



In [19]:

# 3) Code interpreter agent (uses code interpreter tool from crewai)
# code_interpreter_tool = CodeInterpreterTool(unsafe_mode=True)

code_execution_agent = Agent(
    role="Senior Code Execution Expert",
    goal="Review and execute the generated Python code by code writer agent to visualize stock data and fix any errors encountered. It can delegate tasks to code writer agent if needed.",
    backstory="You are a code execution expert. You are skilled at executing Python code.",
    # tools=[code_interpreter_tool],
    allow_code_execution=True,   # This automatically adds the CodeInterpreterTool
    allow_delegation=True,
    llm=llm,
    verbose=True,
)


FileNotFoundError: [Errno 2] No such file or directory: '/usr/bin/docker'

In [20]:

code_execution_task = Task(
    description="""Review and execute the generated Python code by code writer agent to visualize stock data and fix any errors encountered.""",
    expected_output="A clean, working and executable Python script file (.py) for stock visualization.",
    agent=code_execution_agent,
)


In [16]:

# Create the crew
crew = Crew(
    agents=[query_parser_agent, code_writer_agent, code_execution_agent],
    tasks=[query_parsing_task, code_writer_task, code_execution_task],
    process=Process.sequential
)


In [17]:

# Function to be wrapped inside MCP tool
def run_financial_analysis(query):
    result = crew.kickoff(inputs={"query": query})
    return result.raw


In [18]:
query = "Plot YTD stock gain of Tesla"
result = run_financial_analysis(query)
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Task: Analyze the user query and extract stock details.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "symbols": ["TSLA"],                                                                                         │
│    "timeframe": "YTD",                                                                                          │
│    "action": "Plot stock gain"                                                                                  │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Task: Write Python code to visualize stock data based on the inputs from the stock analyst                     │
│                     where you would find stock symbol, timeframe and action.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│  import datetime                                                                                                │
│                                                                                                                 │
│  def visualize_stock_data(symbols, timeframe, action):                                                          │
│      # Fetch the current date and define the start date based on the timeframe                                  │
│      end_date = datetime.datetime.now()                                                                         │
│      if timeframe.lower() == "ytd":                                                                             │
│          start_date = datetime.datetime(end_date.year, 1, 1)                                                    │
│      else:                                                                                                      │
│          raise ValueError("Unsupported timeframe. Please use 'YTD'.")                                           │
│                                                                                                                 │
│      # Download the stock data                                                                                  │
│      stock_data = yf.download(symbols, start=start_date, end=end_date)                                          │
│                                                                                                                 │
│      # Check the action specified and perform the corresponding visualization                                   │
│      if action.lower() == "plot stock gain":                                                                    │
│          # Calculate the stock gain                                                                             │
│          stock_data['Gain'] = (stock_data['Close'] - stock_data['Open']) / stock_data['Open'] * 100             │
│                                                                                                                 │
│          # Plotting the Gain                                                                                    │
│          plt.figure(figsize=(12, 6))                                                                            │
│          plt.plot(stock_data.index, stock_data['Gain'], label=symbols[0] + ' Gain (%)', color='blue')           │
│          plt.title(f'Stock Gain for {symbols[0]} YTD')                                                          │
│          plt.xlabel('Date')                                                                                     │
│          plt.ylabel('Gain (%)')                                                                                 │
│          plt.axhline(0, color='red', linewidth=0.8, linestyle='--')                                             │
│          plt.legend()                                  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Task: Review and execute the generated Python code by code writer agent to visualize stock data and fix any    │
│  errors encountered.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Thought: It seems that there is an issue with executing the code directly, and I might not have access to the  │
│  required libraries in the environment for visualization. Therefore, I will delegate the task of checking the   │
│  stock visualization functionality to a Senior Python Developer to ensure there are no hidden issues in the     │
│  original code before it can be finalized.                                                                      │
│                                                                                                                 │
│  Using Tool: Delegate work to coworker                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"task\": \"Review and execute the provided Python code to visualize stock data and check for any errors or  │
│  issues that need fixing before finalization.\", \"context\": \"The code fetches stock data for TSLA year to    │
│  date (YTD) using yfinance, calculates the percentage gain, and visualizes it using matplotlib. It checks for   │
│  the supported action and timeframe before proceeding with fetching the data and plotting.\", \"coworker\":     │
│  \"Senior Python Developer\"}"                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  ```python                                                                                                      │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│                                                                                                                 │
│  # Parameters                                                                                                   │
│  symbol = 'TSLA'                                                                                                │
│  start_date = '2023-01-01'                                                                                      │
│  end_date = pd.to_datetime('today').strftime('%Y-%m-%d')                                                        │
│                                                                                                                 │
│  # Fetch stock data                                                                                             │
│  def fetch_stock_data(symbol, start, end):                                                                      │
│      stock_data = yf.download(symbol, start=start, end=end)                                                     │
│      return stock_data                                                                                          │
│                                                                                                                 │
│  # Calculate percentage gain                                                                                    │
│  def calculate_percentage_gain(stock_data):                                                                     │
│      stock_data['Percentage Gain'] = ((stock_data['Close'] - stock_data['Close'].iloc[0]) /                     │
│  stock_data['Close'].iloc[0]) * 100                                                                             │
│      return stock_data                                                                                          │
│                                                                                                                 │
│  # Visualize the stock data                                                                                     │
│  def visualize_stock_data(stock_data):                                                                          │
│      plt.figure(figsize=(12, 6))                                                                                │
│      plt.plot(stock_data.index, stock_data['Percentage Gain'], label='Percentage Gain', color='blue')           │
│      plt.title(f'{symbol} Stock Percentage Gain YTD')                                                           │
│      plt.xlabel('Date')                                                                                         │
│      plt.ylabel('Percentage Gain (%)')                                                                          │
│      plt.axhline(0, color='grey', linewidth=0.8, linestyle='--')                                                │
│      plt.grid()                                                                                                 │
│      plt.legend()                                                                                               │
│      plt.tight_layout()                                                                                         │
│      plt.show()                                       

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│                                                                                                                 │
│  # Parameters                                                                                                   │
│  symbol = 'TSLA'                                                                                                │
│  start_date = '2023-01-01'                                                                                      │
│  end_date = pd.to_datetime('today').strftime('%Y-%m-%d')                                                        │
│                                                                                                                 │
│  # Fetch stock data                                                                                             │
│  def fetch_stock_data(symbol, start, end):                                                                      │
│      stock_data = yf.download(symbol, start=start, end=end)                                                     │
│      return stock_data                                                                                          │
│                                                                                                                 │
│  # Calculate percentage gain                                                                                    │
│  def calculate_percentage_gain(stock_data):                                                                     │
│      stock_data['Percentage Gain'] = ((stock_data['Close'] - stock_data['Close'].iloc[0]) /                     │
│  stock_data['Close'].iloc[0]) * 100                                                                             │
│      return stock_data                                                                                          │
│                                                                                                                 │
│  # Visualize the stock data                                                                                     │
│  def visualize_stock_data(stock_data):                                                                          │
│      plt.figure(figsize=(12, 6))                                                                                │
│      plt.plot(stock_data.index, stock_data['Percentage Gain'], label='Percentage Gain', color='blue')           │
│      plt.title(f'{symbol} Stock Percentage Gain YTD')                                                           │
│      plt.xlabel('Date')                                                                                         │
│      plt.ylabel('Percentage Gain (%)')                                                                          │
│      plt.axhline(0, color='grey', linewidth=0.8, linestyle='--')                                                │
│      plt.grid()                                        

```python
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Parameters
symbol = 'TSLA'
start_date = '2023-01-01'
end_date = pd.to_datetime('today').strftime('%Y-%m-%d')

# Fetch stock data
def fetch_stock_data(symbol, start, end):
    stock_data = yf.download(symbol, start=start, end=end)
    return stock_data

# Calculate percentage gain
def calculate_percentage_gain(stock_data):
    stock_data['Percentage Gain'] = ((stock_data['Close'] - stock_data['Close'].iloc[0]) / stock_data['Close'].iloc[0]) * 100
    return stock_data

# Visualize the stock data
def visualize_stock_data(stock_data):
    plt.figure(figsize=(12, 6))
    plt.plot(stock_data.index, stock_data['Percentage Gain'], label='Percentage Gain', color='blue')
    plt.title(f'{symbol} Stock Percentage Gain YTD')
    plt.xlabel('Date')
    plt.ylabel('Percentage Gain (%)')
    plt.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.s

In [ ]:

if __name__ == "__main__":
    # Run the crew with a query
    # query = input("Enter the stock to analyze: ")
    query = "Plot YTD stock gain of Tesla"
    result = run_financial_analysis(query)
    print(result)